In [1]:
from openai import OpenAI
import json
import os
from dotenv import load_dotenv

# 加载 .env 文件里面的密钥
load_dotenv()

api_key = os.getenv("LLM_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)
print("✅ 客户端初始化成功")

✅ 客户端初始化成功


In [2]:
import random

def get_weather(city):
    """查询指定城市的天气（模拟，实际可接真实天气 API）"""
    # 模拟不同城市的天气
    weather_db = {
        "北京": "晴天，25°C",
        "上海": "多云，28°C",
        "广州": "雷阵雨，30°C",
        "深圳": "阴天，29°C",
        "杭州": "小雨，24°C"
    }
    
    if city in weather_db:
        return f"{city}今天{weather_db[city]}"
    else:
        # 随机生成，让演示更有趣
        temps = [20, 22, 25, 28, 30, 32]
        weathers = ["晴天", "多云", "阴天", "小雨"]
        return f"{city}今天{random.choice(weathers)}，{random.choice(temps)}°C"

def calculate(expression):
    """计算数学表达式（安全版 eval）"""
    try:
        # 只允许数字和运算符，防止危险代码
        allowed_chars = set("0123456789+-*/(). ")
        if not all(c in allowed_chars for c in expression):
            return "错误：表达式包含非法字符"
        
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"计算错误：{str(e)}"

def search_web(query):
    """模拟网页搜索（实际可接 SerpAPI / DuckDuckGo）"""
    # 模拟搜索结果
    knowledge = {
        "Python": "Python 是一种高级编程语言，由 Guido van Rossum 于 1991 年创建。",
        "人工智能": "人工智能（AI）是计算机科学的一个分支，致力于创造能够执行通常需要人类智能的任务的系统。",
        "机器学习": "机器学习是 AI 的子集，通过数据训练模型，使计算机能够自动改进性能。",
        "数据科学": "数据科学是一门跨学科领域，使用科学方法、流程、算法和系统从数据中提取知识。"
    }
    
    for key, value in knowledge.items():
        if key in query:
            return f"关于'{query}'的搜索结果：{value}"
    
    return f"关于'{query}'的搜索结果：找到多个相关网页，建议查看维基百科或相关文档。"

# 测试工具
print(get_weather("北京"))
print(calculate("23 * 47"))
print(search_web("什么是Python"))

北京今天晴天，25°C
23 * 47 = 1081
关于'什么是Python'的搜索结果：Python 是一种高级编程语言，由 Guido van Rossum 于 1991 年创建。


In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的当前天气情况，包括温度和天气状况",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，例如：北京、上海、广州"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "计算数学表达式的结果，支持加减乘除和括号",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式，例如：23 * 47、15.5 + 30.2"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "搜索互联网获取某个主题的信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "搜索关键词"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print(f"✅ 已定义 {len(tools)} 个工具")
for t in tools:
    print(f"  - {t['function']['name']}: {t['function']['description']}")

✅ 已定义 3 个工具
  - get_weather: 获取指定城市的当前天气情况，包括温度和天气状况
  - calculate: 计算数学表达式的结果，支持加减乘除和括号
  - search_web: 搜索互联网获取某个主题的信息


In [4]:
def run_agent_step(user_input, messages_history=None):
    """单步 Agent：用户输入 → AI 决策"""
    if messages_history is None:
        messages = [{"role": "system", "content": "你是一个智能助手，可以调用工具帮助用户。如果不需要工具，直接回答。"}]
    else:
        messages = messages_history.copy()
    
    messages.append({"role": "user", "content": user_input})
    
    # 第一次调用：让 AI 决定
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        tools=tools,
        tool_choice="auto"  # auto = AI 自己决定，none = 强制不用工具
    )
    
    return response, messages

# 测试 1：不需要工具的问题
print("=== 测试1：不需要工具 ===")
response, msgs = run_agent_step("你好，请介绍一下自己")
print(f"AI 回复: {response.choices[0].message.content}")
print(f"是否调用工具: {response.choices[0].message.tool_calls is not None}")

# 测试 2：需要工具的问题
print("\n=== 测试2：需要查天气 ===")
response, msgs = run_agent_step("北京今天天气怎么样？")
print(f"AI 回复: {response.choices[0].message.content}")
print(f"是否调用工具: {response.choices[0].message.tool_calls is not None}")

if response.choices[0].message.tool_calls:
    tool_call = response.choices[0].message.tool_calls[0]
    print(f"要调用的工具: {tool_call.function.name}")
    print(f"参数: {tool_call.function.arguments}")

=== 测试1：不需要工具 ===
AI 回复: 你好！我是一个智能助手，很高兴认识你！🌟

我可以帮助你做很多事情，以下是我的主要能力：

**🌤️ 天气查询**
- 我可以查询指定城市的当前天气情况，包括温度和天气状况

**🧮 数学计算**
- 我可以帮你计算各种数学表达式，支持加减乘除和括号运算，比如 `23 × 47` 或 `15.5 + 30.2`

**🔍 网络搜索**
- 我可以搜索互联网，帮你获取某个主题的最新信息和资料

**💬 对话交流**
- 当然，我也擅长日常对话，可以回答你的一般性问题，提供建议和帮助

我的目标是尽力为你提供准确、有用的帮助。如果你有任何疑问或需要帮助的地方，随时告诉我！请问今天有什么可以帮到你的吗？😊
是否调用工具: False

=== 测试2：需要查天气 ===
AI 回复: 我来帮你查询北京今天的天气情况。
是否调用工具: True
要调用的工具: get_weather
参数: {"city": "北京"}


In [9]:
# 加载密钥
load_dotenv()
api_key = os.getenv("LLM_API_KEY")
class ToolAgent:
    def __init__(self, api_key, system_prompt=None):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.deepseek.com"
        )
        self.tools = tools
        self.messages = []
        
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})
        else:
            self.messages.append({
                "role": "system", 
                "content": "你是一个智能助手。当用户需要天气、计算或搜索时，调用相应工具。回答要简洁自然。"
            })
    
    def chat(self, user_input):
        """完整的对话循环"""
        # 1. 记录用户输入
        self.messages.append({"role": "user", "content": user_input})
        print(f"\n🧑 用户: {user_input}")
        
        # 2. 第一次调用：AI 决策
        response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=self.messages,
            tools=self.tools,
            tool_choice="auto"
        )
        
        assistant_message = response.choices[0].message
        
        # 3. 检查是否需要调用工具
        if assistant_message.tool_calls:
            # AI 要求调用工具
            print(f"🤖 AI 决定调用工具...")
            
            # 把 AI 的"工具调用请求"加入对话历史
            self.messages.append({
                "role": "assistant",
                "content": assistant_message.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": tc.type,
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    } for tc in assistant_message.tool_calls
                ]
            })
            
            # 4. 执行所有请求的工具
            for tool_call in assistant_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)
                
                print(f"   🔧 调用 {func_name}({func_args})")
                
                # 执行函数
                if func_name == "get_weather":
                    result = get_weather(**func_args)
                elif func_name == "calculate":
                    result = calculate(**func_args)
                elif func_name == "search_web":
                    result = search_web(**func_args)
                else:
                    result = f"未知工具: {func_name}"
                
                print(f"   📤 工具返回: {result}")
                
                # 5. 把工具结果以特殊角色 tool 返回给 AI
                self.messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result)
                })
            
            # 6. 第二次调用：AI 基于工具结果生成最终回答
            final_response = self.client.chat.completions.create(
                model="deepseek-chat",
                messages=self.messages
            )
            
            final_reply = final_response.choices[0].message.content
            self.messages.append({"role": "assistant", "content": final_reply})
            print(f"🤖 AI: {final_reply}")
            return final_reply
            
        else:
            # 不需要工具，直接回答
            reply = assistant_message.content
            self.messages.append({"role": "assistant", "content": reply})
            print(f"🤖 AI: {reply}")
            return reply
    
    def show_history(self):
        """查看完整对话历史（调试用）"""
        print(f"\n{'='*40}")
        print(f"对话历史（共 {len(self.messages)} 条）:")
        for i, msg in enumerate(self.messages):
            role = msg["role"]
            content = msg.get("content", "")[:60]
            print(f"{i}. [{role}] {content}...")
        print(f"{'='*40}")

# 创建 Agent 实例（不再明文写密钥）
agent = ToolAgent(api_key=api_key)

In [10]:
agent.chat("北京今天天气怎么样？")


🧑 用户: 北京今天天气怎么样？
🤖 AI 决定调用工具...
   🔧 调用 get_weather({'city': '北京'})
   📤 工具返回: 北京今天晴天，25°C
🤖 AI: 北京今天天气晴朗，气温25°C，适合外出活动。


'北京今天天气晴朗，气温25°C，适合外出活动。'

In [11]:
agent.chat("帮我算一下 156 除以 12 再加 38 等于多少？")


🧑 用户: 帮我算一下 156 除以 12 再加 38 等于多少？
🤖 AI 决定调用工具...
   🔧 调用 calculate({'expression': '156 / 12 + 38'})
   📤 工具返回: 156 / 12 + 38 = 51.0
🤖 AI: 计算结果为 **51**。


'计算结果为 **51**。'

In [12]:
agent.chat("什么是机器学习？")


🧑 用户: 什么是机器学习？
🤖 AI 决定调用工具...
   🔧 调用 search_web({'query': '机器学习是什么 定义 介绍'})
   📤 工具返回: 关于'机器学习是什么 定义 介绍'的搜索结果：机器学习是 AI 的子集，通过数据训练模型，使计算机能够自动改进性能。
🤖 AI: 机器学习是人工智能（AI）的一个子集，核心是让计算机通过数据训练模型，自动从中学习规律并改进性能，而无需显式编程。简单来说，就是让机器从经验中“自学”来完成任务。


'机器学习是人工智能（AI）的一个子集，核心是让计算机通过数据训练模型，自动从中学习规律并改进性能，而无需显式编程。简单来说，就是让机器从经验中“自学”来完成任务。'

In [13]:
agent.chat("谢谢你的帮助！")


🧑 用户: 谢谢你的帮助！
🤖 AI: 不客气！如果以后需要查天气、算数学题或搜索信息，随时找我哦！😊


'不客气！如果以后需要查天气、算数学题或搜索信息，随时找我哦！😊'

In [14]:
agent2 = ToolAgent(api_key=api_key)
agent2.chat("如果北京今天气温是30度，广州是35度，两地温差是多少？")


🧑 用户: 如果北京今天气温是30度，广州是35度，两地温差是多少？
🤖 AI 决定调用工具...
   🔧 调用 get_weather({'city': '北京'})
   📤 工具返回: 北京今天晴天，25°C
   🔧 调用 get_weather({'city': '广州'})
   📤 工具返回: 广州今天雷阵雨，30°C
🤖 AI: 根据实时天气数据，北京现在气温是25°C，广州是30°C，两地温差为5°C（广州比北京高5度）。您提到的30度和35度可能不是当前数据，建议以实时信息为准。


'根据实时天气数据，北京现在气温是25°C，广州是30°C，两地温差为5°C（广州比北京高5度）。您提到的30度和35度可能不是当前数据，建议以实时信息为准。'

In [15]:
# 使用同一个 agent，继续对话
agent.chat("那上海呢？")  # 应该能理解"那"指的是天气
agent.chat("帮我算一下刚才那个温差的平方根")


🧑 用户: 那上海呢？
🤖 AI 决定调用工具...
   🔧 调用 get_weather({'city': '上海'})
   📤 工具返回: 上海今天多云，28°C
🤖 AI: 上海今天多云，气温28°C。需要其他城市的天气也可以告诉我！

🧑 用户: 帮我算一下刚才那个温差的平方根
🤖 AI 决定调用工具...
   🔧 调用 calculate({'expression': 'sqrt(3)'})
   📤 工具返回: 错误：表达式包含非法字符
🤖 AI: 抱歉，刚才的表达式格式有误，我重新计算一下。

<｜｜DSML｜｜tool_calls>
<｜｜DSML｜｜invoke name="calculate">
<｜｜DSML｜｜parameter name="expression" string="true">sqrt(3)</｜｜DSML｜｜parameter>
</｜｜DSML｜｜invoke>
</｜｜DSML｜｜tool_calls>


'抱歉，刚才的表达式格式有误，我重新计算一下。\n\n<｜｜DSML｜｜tool_calls>\n<｜｜DSML｜｜invoke name="calculate">\n<｜｜DSML｜｜parameter name="expression" string="true">sqrt(3)</｜｜DSML｜｜parameter>\n</｜｜DSML｜｜invoke>\n</｜｜DSML｜｜tool_calls>'

In [16]:
agent.show_history()


对话历史（共 23 条）:
0. [system] 你是一个智能助手。当用户需要天气、计算或搜索时，调用相应工具。回答要简洁自然。...
1. [user] 北京今天天气怎么样？...
2. [assistant] 我来帮你查询北京的天气情况。...
3. [tool] 北京今天晴天，25°C...
4. [assistant] 北京今天天气晴朗，气温25°C，适合外出活动。...
5. [user] 帮我算一下 156 除以 12 再加 38 等于多少？...
6. [assistant] 我来帮你计算这个表达式。...
7. [tool] 156 / 12 + 38 = 51.0...
8. [assistant] 计算结果为 **51**。...
9. [user] 什么是机器学习？...
10. [assistant] 让我帮你搜索一下机器学习的相关信息。...
11. [tool] 关于'机器学习是什么 定义 介绍'的搜索结果：机器学习是 AI 的子集，通过数据训练模型，使计算机能够自动改进性能。...
12. [assistant] 机器学习是人工智能（AI）的一个子集，核心是让计算机通过数据训练模型，自动从中学习规律并改进性能，而无需显式编程。简单来...
13. [user] 谢谢你的帮助！...
14. [assistant] 不客气！如果以后需要查天气、算数学题或搜索信息，随时找我哦！😊...
15. [user] 那上海呢？...
16. [assistant] 你是想问上海的天气吗？我来帮你查一下。...
17. [tool] 上海今天多云，28°C...
18. [assistant] 上海今天多云，气温28°C。需要其他城市的天气也可以告诉我！...
19. [user] 帮我算一下刚才那个温差的平方根...
20. [assistant] 你是想计算上海（28°C）和北京（25°C）之间的温差，然后求平方根吧？

温差是 28 - 25 = 3，我来计算一下...
21. [tool] 错误：表达式包含非法字符...
22. [assistant] 抱歉，刚才的表达式格式有误，我重新计算一下。

<｜｜DSML｜｜tool_calls>
<｜｜DSML｜｜invoke...


## 📝 智能工具调用助手（AI Agent）总结

### 1. 项目目标
实现一个能理解用户意图、自动调用外部工具（天气/计算/搜索）并整合结果的 AI Agent。

### 2. 处理流程（核心：ReAct 模式）


### 3. 核心组件

| 组件 | 作用 | 代码体现 |
|------|------|---------|
| **工具函数** | 执行具体任务 | `get_weather()`, `calculate()`, `search_web()` |
| **工具描述** | 告诉 AI 有什么工具、怎么用 | `tools` JSON Schema 列表 |
| **Agent 循环** | 连接推理和执行的桥梁 | `ToolAgent.chat()` 方法 |
| **对话历史** | 维护多轮上下文 | `self.messages` 列表 |

### 4. 关键发现
- **AI 能自主决策**：问"北京天气"时自动调用 `get_weather`，问"你好"时不调用
- **参数自动提取**：用户说"北京"，AI 自动填入 `{"city": "北京"}`
- **多轮工具链**：复杂问题可能触发多次工具调用（先搜索再计算）
- **工具角色特殊**：`role="tool"` 是 OpenAI/DeepSeek 的特殊消息类型，必须包含 `tool_call_id`

### 5. 与第五周聊天机器人的区别
| | 第五周聊天机器人 | 第八周 AI Agent |
|--|----------------|----------------|
| 能力 | 只回答文本问题 | 能执行外部操作（查天气、算数、搜索） |
| 机制 | 纯对话历史 | 对话历史 + 工具调用循环 |
| 消息角色 | system/user/assistant | system/user/assistant/**tool** |
| 应用场景 | 问答、陪伴 | 自动化任务、信息查询、计算辅助 |

### 6. 局限与风险
- **工具参数可能出错**：AI 可能把"北京市"传成"北京"，需要后处理兼容
- **幻觉调用**：AI 可能错误地决定调用不相关的工具
- **延迟**：每次工具调用需要 2 次 API 请求（决策 + 总结），响应更慢
- **安全问题**：如果工具是 `eval()` 或文件删除，必须严格校验参数

### 7. 下一步优化
- 接入真实天气 API（OpenWeatherMap）
- 接入真实搜索引擎（SerpAPI）
- 加入更多工具：发邮件、查日历、操作 Excel
- 实现多 Agent 协作（一个规划，一个执行）